# Controlled experiment: recency weighting для Pilot v4 LightGBM

Проверяем, станет ли tuned primary LightGBM точнее, если более свежие train observations сильнее влияют на loss. **Recency weighting не создаёт новый feature:** в матрице `X` по-прежнему только `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES`; меняется лишь `sample_weight` при `fit`.

Гипотеза разумна для нестационарного пассажиропотока: режим движения, сезонный спрос и привычки пассажиров могут меняться. Но более сильный вес свежих дат может одновременно ухудшить переносимость на другой сезон. Поэтому сравниваем три заранее выбранных схемы с reference на одинаковых temporal folds. Ни HPO весов, ни Jan–Oct → Nov–Dec fit, ни нового submission здесь нет.

## 1. Исторические данные и frozen model contract

Напрямую используем те же model-ready historical CSV и HPO result, что и model selection. Скрипт проверяет SHA-256 данных и кода, выбранные features, параметры tuned LightGBM и даты folds. Frozen остаются `regression_l1`, 400 деревьев, `max_depth=8`, `learning_rate=0.1`, `num_leaves=15`, `min_child_samples=40`, random seed, categorical handling, postprocessing и правило route 5. Forecast Nov–Dec не читается.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'ml/src/recency_weighting_experiment.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(root / 'ml/src'))
import recency_weighting_experiment as rw
import model_selection as ms

rows, folds, hpo = rw.load_inputs()
parameters = hpo['final_configuration']['parameters']
display(pd.DataFrame([{'historical_rows': len(rows), 'target': 'boardings',
                       'model': 'LightGBM',
                       'feature_count': len(ms.FEATURE_CONFIGURATIONS[rw.CONFIGURATION]),
                       'fold_count': len(folds), 'data_and_code_hashes_match': True}]))

,historical_rows,target,model,feature_count,fold_count,data_and_code_hashes_match
0,72960,boardings,LightGBM,12,3,True


## 2. Temporal folds и момент расчёта весов

Используем Jan–Apr → May–Jun, Jan–Jun → Jul–Aug и Jan–Aug → Sep–Oct. Для **каждого fold отдельно** `age_days = max(fold_train.date) − train.date`. Поэтому первая train origin — 30 апреля, вторая — 30 июня, третья — 31 августа. Применение августовской или октябрьской даты к ранним folds дало бы неверные веса и нарушило temporal experiment. Random split не используется.

Основная метрика — pooled WAPE: сумма абсолютных ошибок трёх непересекающихся holdouts, делённая на сумму фактических посадок. Sep–Oct не должен единолично определять схему весов.

In [2]:
display(pd.DataFrame([{'fold': spec[0], 'train_end': train.date.max().date(),
                       'validation_start': valid.date.min().date(),
                       'validation_end': valid.date.max().date(),
                       'train_rows': len(train), 'validation_rows': len(valid)}
                      for spec, (train, valid) in zip(ms.FOLDS, folds)]))

,fold,train_end,validation_start,validation_end,train_rows,validation_rows
0,jan_apr_to_may_jun,2025-04-30,2025-05-01,2025-06-30,28800,14640
1,jan_jun_to_jul_aug,2025-06-30,2025-07-01,2025-08-31,43440,14880
2,jan_aug_to_sep_oct,2025-08-31,2025-09-01,2025-10-31,58320,14640


## 3. Reference без weighting — обязательная остановка при расхождении

Configuration A вызывает тот же fit/predict код, что и остальные варианты, но с `sample_weight=None`. До проверки weighting сверяем WAPE, absolute error и per-route absolute error с tuned LightGBM из HPO. При расхождении notebook остановится здесь. После `predict` применяются те же округление и принудительный `0` для route 5.

In [3]:
all_folds = {'reference': rw.evaluate_configuration(folds, 'reference', parameters)}
rw.assert_reference_reproduced(all_folds['reference'], hpo)
display(pd.DataFrame([{'fold': spec[0], 'reference WAPE': actual['wape'],
                       'HPO WAPE': prior['wape'],
                       'absolute_error_delta': actual['absolute_error'] - prior['absolute_error']}
                      for spec, actual, prior in zip(
                          ms.FOLDS, all_folds['reference'],
                          hpo['studies']['LightGBM']['best_tuned']['fold_results'])]).round(9))
print('Reference pooled WAPE:', round(ms.summarize(all_folds['reference'])['pooled_wape'], 9))

,fold,reference WAPE,HPO WAPE,absolute_error_delta
0,jan_apr_to_may_jun,0.141207,0.141207,0
1,jan_jun_to_jul_aug,0.205422,0.205422,0
2,jan_aug_to_sep_oct,0.114021,0.114021,0


Reference pooled WAPE: 0.150956111


## 4. Три заранее заданных схемы весов

**B, fixed:** последние 0–30 дней получают `1.00`, 31–60 — `0.95`, 61–120 — `0.85`, более старые — `0.70`. Это ступенчатое снижение.

**C/D, exponential:** `exp(-ln(2) × age_days / half_life_days)`. Half-life — возраст, при котором вес уменьшается до половины веса самой свежей строки. Заранее проверяем только 120 и 180 дней: это умеренные горизонты памяти около четырёх и шести месяцев для короткой истории 2025 года. Другие веса и half-life не подбираем по Sep–Oct.

Таблица показывает агрегированные веса каждого fold и его собственную origin. Она помогает проверить, что нет отрицательного возраста, нулевых/невалидных весов или использования будущей даты.

In [4]:
weight_rows = []
for name in rw.WEIGHTING_CONFIGURATIONS:
    for spec, (train, _) in zip(ms.FOLDS, folds):
        _, diagnostic = rw.sample_weights(train, name)
        weight_rows.append({'weighting': name, 'fold': spec[0],
                            'origin': diagnostic['origin'],
                            'max_age_days': diagnostic['max_age_days'],
                            'min_weight': diagnostic['min_weight'],
                            'mean_weight': diagnostic['mean_weight'],
                            'max_weight': diagnostic['max_weight']})
display(pd.DataFrame(weight_rows).round(4))

,weighting,fold,origin,max_age_days,min_weight,mean_weight,max_weight
0,reference,jan_apr_to_may_jun,2025-04-30,119,1.0000,1.0000,1.0
1,reference,jan_jun_to_jul_aug,2025-06-30,180,1.0000,1.0000,1.0
2,reference,jan_aug_to_sep_oct,2025-08-31,242,1.0000,1.0000,1.0
3,fixed,jan_apr_to_may_jun,2025-04-30,119,0.8500,0.9137,1.0
4,fixed,jan_jun_to_jul_aug,2025-06-30,180,0.7000,0.8425,1.0
5,fixed,jan_aug_to_sep_oct,2025-08-31,242,0.7000,0.8062,1.0
6,exponential_120,jan_apr_to_may_jun,2025-04-30,119,0.5029,0.7234,1.0
7,exponential_120,jan_jun_to_jul_aug,2025-06-30,180,0.3536,0.6221,1.0
8,exponential_120,jan_aug_to_sep_oct,2025-08-31,242,0.2471,0.5389,1.0
9,exponential_180,jan_apr_to_may_jun,2025-04-30,119,0.6324,0.8023,1.0


## 5. Обучение B/C/D на тех же folds

Для всех трёх вариантов код обучения, preprocessing, модель, признаки, validation строки и метрики совпадают с A. В `fit` меняется только массив `sample_weight`. Веса вычисляются заново из train даты текущего fold; validation target становится доступен лишь при расчёте ошибки после prediction. Выводим компактную таблицу вместо raw LightGBM logs.

In [5]:
for name in rw.WEIGHTING_CONFIGURATIONS[1:]:
    all_folds[name] = rw.evaluate_configuration(folds, name, parameters)
result = rw.build_result(all_folds, hpo)

comparison = []
baseline = result['summaries']['reference']['pooled_wape']
for name in rw.WEIGHTING_CONFIGURATIONS:
    summary = result['summaries'][name]
    comparison.append({'Weighting': name,
                       'May–Jun': all_folds[name][0]['wape'],
                       'Jul–Aug': all_folds[name][1]['wape'],
                       'Sep–Oct': all_folds[name][2]['wape'],
                       'Pooled WAPE': summary['pooled_wape'],
                       'Δ vs reference': summary['pooled_wape'] - baseline,
                       'Pooled AE': summary['pooled_absolute_error']})
display(pd.DataFrame(comparison).round(6))

,Weighting,May–Jun,Jul–Aug,Sep–Oct,Pooled WAPE,Δ vs reference,Pooled AE
0,reference,0.141207,0.205422,0.114021,0.150956,0.000000,5287697
1,fixed,0.144850,0.203475,0.114953,0.151902,0.000946,5320819
2,exponential_120,0.146425,0.199075,0.118874,0.152503,0.001546,5341867
3,exponential_180,0.144429,0.200580,0.116781,0.151542,0.000586,5308233


## 6. Устойчивость, маршруты и решение

Pooled WAPE — главный критерий, но обязательно смотрим все три folds и маршруты. `RECENCY_WEIGHTING_SUPPORTED` требует pooled выигрыша, отсутствия ухудшения любого fold и отсутствия route-level ухудшения более 0.01 абсолютного WAPE. Если pooled WAPE улучшился, но folds или маршруты ведут себя неоднородно, это `MIXED`; если ни один вариант не улучшил pooled WAPE, это `REJECTED`. Такое правило не позволяет объявить успех по одному Sep–Oct.

Route 5 имеет нулевой фактический знаменатель: его prediction всегда `0`, а per-route WAPE не определяется. Ниже сравниваются reference и лучший по pooled WAPE **weighted** candidate, даже если он хуже reference.

In [6]:
details = []
for name in rw.WEIGHTING_CONFIGURATIONS:
    summary = result['summaries'][name]
    details.append({'weighting': name, 'WAPE-score': summary['wape_score'],
                    'mean fold WAPE': summary['mean_wape'],
                    'median fold WAPE': summary['median_wape'],
                    'fold range': summary['fold_wape_range'],
                    'fit seconds': summary['train_seconds'],
                    'predict seconds': summary['inference_seconds']})
display(pd.DataFrame(details).round(6))

best = result['best_weighting']
route_rows = []
for route in ms.ROUTES:
    base = result['summaries']['reference']['per_route'][str(route)]['wape']
    candidate = result['summaries'][best]['per_route'][str(route)]['wape']
    route_rows.append({'route': route, 'reference WAPE': base,
                       f'{best} WAPE': candidate,
                       'Δ WAPE': None if base is None else candidate - base})
display(pd.DataFrame(route_rows).round(6))
print('Best weighted candidate:', best,
      '| pooled delta:', round(result['best_delta_pooled_wape'], 6),
      '| classification:', result['classification'])

,weighting,WAPE-score,mean fold WAPE,median fold WAPE,fold range,fit seconds,predict seconds
0,reference,0.849044,0.153550,0.141207,0.091401,2.648280,0.592488
1,fixed,0.848098,0.154426,0.144850,0.088522,16.295844,0.631298
2,exponential_120,0.847497,0.154791,0.146425,0.080200,16.057249,0.640573
3,exponential_180,0.848458,0.153930,0.144429,0.083800,15.384987,0.582895


,route,reference WAPE,exponential_180 WAPE,Δ WAPE
0,1,0.124881,0.124882,0.000001
1,5,NaN,NaN,NaN
2,7,0.271423,0.272103,0.000681
3,11,0.126133,0.125244,-0.000889
4,12,0.121498,0.118813,-0.002685
5,17,0.101852,0.108467,0.006615
6,25,0.193525,0.190624,-0.002901
7,26,0.165674,0.163202,-0.002471
8,28,0.184384,0.180225,-0.004158
9,50,0.212339,0.211870,-0.000469


Best weighted candidate: exponential_180 | pooled delta: 0.000586 | classification: RECENCY_WEIGHTING_REJECTED


## 7. Сохранение experiment provenance

Сохраняем фиксированные схемы весов, fold origins, метрики, per-route WAPE, времена, параметры модели, версии библиотек и SHA-256 исходных исторических CSV и кода в существующей experiment infrastructure. Notebook и CLI используют один и тот же runner; notebook действительно выполнил все четыре конфигурации. Nov–Dec fit и submission не выполняются.

In [7]:
rw.save_result(result)
print('Saved:', rw.ARTIFACT.relative_to(root).as_posix())
print('Classification:', result['classification'])

Saved: ml/experiments/results/recency_weighting.json
Classification: RECENCY_WEIGHTING_REJECTED


## Вывод

Гипотеза получает классификацию **`RECENCY_WEIGHTING_REJECTED`** по заранее заданной основной метрике: все три weighted-варианта имеют pooled WAPE выше reference `0.150956`. Лучший weighted вариант — exponential half-life 180 дней (`0.151542`, Δ `+0.000586`). Он улучшил Jul–Aug на `−0.004842` WAPE, но ухудшил May–Jun на `+0.003222` и Sep–Oct на `+0.002760`; pooled per-route WAPE сильнее всего ухудшился на route 17 примерно на `+0.006615`. Локальное летнее улучшение не подтверждает общую гипотезу. Текущая unweighted primary LightGBM и accepted feature registry остаются прежними; нового Nov–Dec прогноза нет.

## Дополнение: обучение варианта D на Jan–Oct

По отдельному запросу обучаем вариант D на всей известной истории Jan–Oct. Это исследовательский прогноз на Nov–Dec после завершённого сравнения весов; вывод `RECENCY_WEIGHTING_REJECTED` и статус primary Pilot v4 остаются прежними. Sep–Oct входит в доступную историю для финального обучения. Параметры, признаки и preprocessing берём из зафиксированного HPO contract; единственное отличие от unweighted LightGBM — `sample_weight`. Возраст наблюдений считаем от последней даты Jan–Oct train, поэтому half-life 180 дней привязан к правильной точке отсчёта.

In [8]:
import numpy as np
from lightgbm import LGBMRegressor
from time import perf_counter

assert len(rows) == 72_960
assert rows['date'].max() == pd.Timestamp('2025-10-31')
assert rows['boardings'].notna().all()
weights, weight_info = rw.sample_weights(rows, 'exponential_180')
assert weight_info['origin'] == '2025-10-31'
assert len(weights) == len(rows) and np.isfinite(weights).all()
assert (weights > 0).all() and np.isclose(weights.max(), 1.0)

x_train = ms.make_features(rows, rw.CONFIGURATION, 'LightGBM')
assert x_train.columns.tolist() == list(ms.FEATURE_CONFIGURATIONS[rw.CONFIGURATION])
model_d = LGBMRegressor(**parameters)
fit_start = perf_counter()
model_d.fit(x_train, rows['boardings'], categorical_feature=list(ms.CATEGORICAL),
            sample_weight=weights)
fit_seconds = perf_counter() - fit_start
display(pd.DataFrame([{'train_rows': len(rows), 'weight_origin': weight_info['origin'],
                       'weight_min': weights.min(), 'weight_mean': weights.mean(),
                       'weight_max': weights.max(), 'fit_seconds': fit_seconds}]).round(3))

,train_rows,weight_origin,weight_min,weight_mean,weight_max,fit_seconds
0,72960,2025-10-31,0.311,0.59,1.0,9.064


## Прогноз на Nov–Dec и submission

Forecast загружаем из текущего model-ready CSV; фактических `boardings` за Nov–Dec в нём нет. Применяем те же LightGBM preprocessing и postprocessing, что и при validation: отсекаем отрицательные значения и округляем до ближайшего целого. Для route 5 прогноз принудительно равен нулю. Порядок строк forecast grid сохраняется. Проверки ниже подтверждают корректность формата и значений прогноза, а не его точность.

In [9]:
forecast_path = root / 'data/processed/model_ready_forecast_2025_nov_dec.csv'
forecast = pd.read_csv(forecast_path, sep=';', parse_dates=['date'],
                       dtype={'temperature_bucket': ms.BUCKET_DTYPE})
keys = ['route', 'date', 'hour']
assert len(forecast) == 14_640 and 'boardings' not in forecast.columns
assert forecast.columns.is_unique and not forecast.isna().any().any()
assert not forecast.duplicated(keys).any()
assert set(forecast['route']) == set(ms.ROUTES)
assert set(forecast['hour']) == set(range(24))
assert forecast['date'].min() == pd.Timestamp('2025-11-01')
assert forecast['date'].max() == pd.Timestamp('2025-12-31')
assert set(forecast['date']) == set(pd.date_range('2025-11-01', '2025-12-31'))

x_forecast = ms.make_features(forecast, rw.CONFIGURATION, 'LightGBM')
assert x_forecast.columns.tolist() == x_train.columns.tolist()
predict_start = perf_counter()
raw_prediction = model_d.predict(x_forecast)
predict_seconds = perf_counter() - predict_start
assert np.isfinite(raw_prediction).all()
prediction = np.floor(np.maximum(raw_prediction, 0) + 0.5).astype('int64')
prediction[forecast['route'].eq(5).to_numpy()] = 0

submission = forecast[keys].copy()
submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
submission['prediction'] = prediction
submission_path = root / 'data/submissions/pilot_model_v4_lightgbm_recency_exp180_submission.csv'
submission_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(submission_path, sep=';', index=False, encoding='utf-8')
display(pd.DataFrame([{'forecast_rows': len(forecast), 'predict_seconds': predict_seconds,
                       'predicted_total': int(prediction.sum()), 'zero_count': int((prediction == 0).sum()),
                       'median': float(np.median(prediction)), 'p95': float(np.quantile(prediction, .95)),
                       'p99': float(np.quantile(prediction, .99)), 'max': int(prediction.max())}]).round(3))

,forecast_rows,predict_seconds,predicted_total,zero_count,median,p95,p99,max
0,14640,0.185,12582330,3014,506.0,2950.0,3861.0,5299


## Проверка сохранённого файла

Перечитываем submission с диска и сверяем схему, ключи, порядок строк, покрытие сетки и допустимость predictions. Совпадение с organizer template проверяем по ключам. Успешная проверка подтверждает готовность файла к отправке, но не означает, что weighting улучшил WAPE.

In [10]:
saved = pd.read_csv(submission_path, sep=';')
template = pd.read_csv(root / 'data/raw/test_submission.csv', sep=';')
assert saved.columns.tolist() == ['route', 'date', 'hour', 'prediction']
assert len(saved) == len(forecast) == len(template) == 14_640
assert saved[keys].equals(submission[keys])
assert saved[keys].equals(template[keys])
assert not saved.duplicated(keys).any() and not saved.isna().any().any()
assert saved['route'].nunique() == 10 and saved['hour'].nunique() == 24
assert saved['date'].min() == '2025-11-01' and saved['date'].max() == '2025-12-31'
assert pd.api.types.is_integer_dtype(saved['prediction'])
assert saved['prediction'].ge(0).all()
assert saved.loc[saved['route'].eq(5), 'prediction'].eq(0).all()
assert np.isfinite(saved['prediction'].to_numpy()).all()
assert saved.equals(submission)
display(pd.DataFrame([{'file': str(submission_path.relative_to(root)), 'rows': len(saved),
                       'total': int(saved['prediction'].sum()), 'checks_passed': True}]))

,file,rows,total,checks_passed
0,data\submissions\pilot_model_v4_lightgbm_recen...,14640,12582330,True


## Итог дополнения

Отдельный submission варианта D создан и проверен. Он остаётся исследовательским: controlled temporal validation не поддержала recency weighting, поэтому primary Pilot v4 и accepted feature registry не меняются.